[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/12_gradient_boosting/01_boosting_intuition_adaboost.ipynb)

# Bài 01 — Boosting: học tuần tự từ sai lầm (AdaBoost)

**🎯 Mục tiêu**
- Phân biệt **bagging** (song song, giảm variance) và **boosting** (tuần tự, giảm **bias**).
- Cài đặt **AdaBoost** from scratch với decision stump: trọng số mẫu, trọng số model $\alpha_m$.
- Thấy lỗi train giảm theo hàm mũ; hiểu AdaBoost = tối thiểu **exponential loss**.
- Failure case: AdaBoost rất nhạy với **nhiễu nhãn**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/12_gradient_boosting

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from ml_course.datasets import make_classification_2d
from ml_course.visualization import plot_decision_boundary, set_style

set_style()

## 1. Ý tưởng: một nhóm "học sinh yếu" học nối tiếp

| | Bagging / Random Forest | Boosting |
|---|---|---|
| Thứ tự | các model **độc lập**, train song song | **tuần tự**: model sau tập trung vào lỗi của model trước |
| Base learner | mạnh, variance cao (cây sâu) | **yếu**, bias cao (stump, cây nông) |
| Giảm chủ yếu | variance | **bias** (và một phần variance) |
| Thêm model | không overfit | **có thể** overfit → cần dừng đúng lúc |

**Weak learner** chỉ cần tốt hơn đoán mò một chút (lỗi < 50%). Câu hỏi của Kearns & Valiant (1988): "nhiều model yếu có ghép thành một model mạnh được không?" — AdaBoost (Freund & Schapire, 1997) là câu trả lời "có".

## 2. Thuật toán AdaBoost (nhị phân, $y \in \{-1, +1\}$)

1. Khởi tạo trọng số mẫu $w_i = 1/n$.
2. Với $m = 1, \dots, M$:
   - Train weak learner $h_m$ trên dữ liệu **có trọng số** $w$.
   - Lỗi có trọng số: $\varepsilon_m = \sum_i w_i\,\mathbb 1[h_m(x_i) \ne y_i]$.
   - Trọng số model: $\alpha_m = \frac12\ln\frac{1 - \varepsilon_m}{\varepsilon_m}$ (lỗi càng nhỏ, tiếng nói càng lớn).
   - Cập nhật: $w_i \leftarrow w_i\, e^{-\alpha_m y_i h_m(x_i)}$, rồi chuẩn hoá — mẫu bị **sai** được tăng trọng số $e^{\alpha_m}$, mẫu đúng giảm $e^{-\alpha_m}$.
3. Dự đoán: $H(x) = \operatorname{sign}\left(\sum_m \alpha_m h_m(x)\right)$.

In [ ]:
class AdaBoost:
    def __init__(self, n_estimators=50, max_depth=1):
        self.n_estimators, self.max_depth = n_estimators, max_depth

    def fit(self, X, y):
        ys = np.where(y == 1, 1, -1)
        n = len(y)
        w = np.full(n, 1 / n)
        self.stumps_, self.alphas_, self.errors_, self.weights_history_ = [], [], [], []
        for _ in range(self.n_estimators):
            stump = DecisionTreeClassifier(max_depth=self.max_depth, random_state=0).fit(X, ys, sample_weight=w)
            pred = stump.predict(X)
            err = np.clip(np.sum(w[pred != ys]), 1e-10, 1 - 1e-10)
            alpha = 0.5 * np.log((1 - err) / err)
            w = w * np.exp(-alpha * ys * pred)
            w /= w.sum()
            self.stumps_.append(stump)
            self.alphas_.append(alpha)
            self.errors_.append(err)
            self.weights_history_.append(w.copy())
        return self

    def decision_function(self, X, n_models=None):
        models = list(zip(self.alphas_, self.stumps_))[:n_models]
        return sum(a * s.predict(X) for a, s in models)

    def predict(self, X):
        return (self.decision_function(X) >= 0).astype(int)


X, y = make_classification_2d("moons", n=300, noise=0.25, seed=3)
X_test, y_test = make_classification_2d("moons", n=2000, noise=0.25, seed=4)
ada = AdaBoost(n_estimators=200).fit(X, y)
print("lỗi có trọng số 5 stump đầu:", np.round(ada.errors_[:5], 3))
print("α của 5 stump đầu:          ", np.round(ada.alphas_[:5], 3))

## 3. Trọng số dồn vào những điểm khó

Kích thước điểm tỉ lệ với trọng số $w_i$ sau 1, 5, 20 vòng:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, m in zip(axes, [1, 5, 20]):
    w = ada.weights_history_[m - 1]

    class Partial:
        def predict(self, Z, m=m):
            return (ada.decision_function(Z, m) >= 0).astype(int)

    plot_decision_boundary(Partial(), X, y, ax=ax)
    ax.scatter(X[:, 0], X[:, 1], s=4000 * w, facecolors="none", edgecolors="k", lw=0.6)
    ax.set_title(f"sau {m} stump: train acc = {np.mean(Partial().predict(X) == y):.2f}")
plt.show()

Mỗi stump chỉ là **một đường thẳng song song trục**, nhưng tổng có trọng số của nhiều stump tạo ranh giới phức tạp. Các điểm nằm gần ranh giới (khó) được phóng to dần.

## 4. Lỗi train giảm theo hàm mũ

Định lý (Freund & Schapire): lỗi train của $H$ bị chặn trên bởi $\prod_m 2\sqrt{\varepsilon_m(1 - \varepsilon_m)} \le \exp\left(-2\sum_m \gamma_m^2\right)$ với $\gamma_m = 0.5 - \varepsilon_m$ — miễn mỗi weak learner tốt hơn đoán mò, lỗi train **giảm theo hàm mũ**.

In [ ]:
M = len(ada.alphas_)
train_err = [np.mean((ada.decision_function(X, m) >= 0).astype(int) != y) for m in range(1, M + 1)]
test_err = [np.mean((ada.decision_function(X_test, m) >= 0).astype(int) != y_test) for m in range(1, M + 1)]
bound = np.cumprod([2 * np.sqrt(e * (1 - e)) for e in ada.errors_])
plt.semilogy(range(1, M + 1), np.maximum(train_err, 1e-4), label="lỗi train")
plt.semilogy(range(1, M + 1), test_err, label="lỗi test")
plt.semilogy(range(1, M + 1), bound, "--", label="chặn trên lý thuyết")
plt.xlabel("số stump")
plt.ylabel("lỗi (log)")
plt.legend()
plt.title("AdaBoost: lỗi train giảm rất nhanh; lỗi test giảm rồi đi ngang")
plt.show()
print(f"sau {M} stump: lỗi train = {train_err[-1]:.3f} | lỗi test = {test_err[-1]:.3f} | lỗi test tốt nhất = {min(test_err):.3f} (tại {int(np.argmin(test_err)) + 1} stump)")

## 5. AdaBoost = tối thiểu exponential loss

Friedman, Hastie & Tibshirani (2000) chỉ ra AdaBoost chính là **forward stagewise additive modeling** tối thiểu hoá

$$L(y, F) = e^{-yF(x)}, \qquad F(x) = \sum_m \alpha_m h_m(x)$$

— mỗi vòng thêm một hàm $\alpha_m h_m$ làm giảm loss nhiều nhất, giữ nguyên các hàm cũ. Góc nhìn này mở đường cho **Gradient Boosting** (bài 02): thay exponential loss bằng **bất kỳ** loss khả vi nào.

In [ ]:
margins = np.linspace(-2, 2, 200)
plt.plot(margins, np.exp(-margins), label="exponential (AdaBoost)")
plt.plot(margins, np.log2(1 + np.exp(-margins)), label="log-loss (logistic, /ln2)")
plt.plot(margins, (margins < 0).astype(float), label="0-1 loss")
plt.xlabel("margin y·F(x)")
plt.ylabel("loss")
plt.ylim(0, 5)
plt.legend()
plt.title("Exponential loss phạt cực nặng các điểm sai với margin âm lớn")
plt.show()

## 6. Failure case: nhiễu nhãn

Exponential loss tăng **theo hàm mũ** với margin âm → các điểm bị **gán nhãn sai** (luôn bị dự đoán "sai") nhận trọng số ngày càng lớn, và boosting dồn sức "học thuộc" chúng. Thí nghiệm: lật ngẫu nhiên 0%, 15%, 30% nhãn train, AdaBoost với cây độ sâu 3; đo test accuracy (trên nhãn **sạch**) theo số vòng:

In [ ]:
Xn, yn = make_classification_2d("moons", n=600, noise=0.2, seed=8)
Xn_te, yn_te = make_classification_2d("moons", n=3000, noise=0.2, seed=9)
for rate in [0.0, 0.15, 0.3]:
    flip = np.random.default_rng(1).random(len(yn)) < rate
    yn_noisy = np.where(flip, 1 - yn, yn)
    model = AdaBoost(n_estimators=400, max_depth=3).fit(Xn, yn_noisy)
    accs = {m: np.mean((model.decision_function(Xn_te, m) >= 0).astype(int) == yn_te) for m in [10, 50, 400]}
    w_flip = model.weights_history_[-1][flip].sum()
    print(f"{rate:4.0%} nhãn bị lật: test acc sau 10 / 50 / 400 vòng = " + " / ".join(f"{a:.3f}" for a in accs.values())
          + (f" | {w_flip:.0%} tổng trọng số nằm trên {flip.mean():.0%} số điểm (bị lật)" if rate else ""))

- Nhãn sạch: thêm vòng gần như không hại.
- Có nhiễu: accuracy test **giảm dần** khi thêm vòng (10 vòng tốt hơn hẳn 400 vòng, nhất là khi 30% nhãn bị lật) — model học thuộc nhiễu; gần **một nửa** tổng trọng số dồn vào các điểm bị lật.
- Về lý thuyết, exponential loss nhạy với nhiễu hơn log-loss (Gradient Boosting — bài 02, phạt tuyến tính khi margin âm). Trong thực hành, **mọi** thuật toán boosting đều cần **dừng sớm** (early stopping) và cây nông khi nhãn nhiễu.

## 7. scikit-learn

In [ ]:
sk_ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=200, random_state=0).fit(X, y)
print(f"AdaBoostClassifier (sklearn): test acc = {sk_ada.score(X_test, y_test):.3f} | scratch: {np.mean(ada.predict(X_test) == y_test):.3f}")

## 🧠 Kiểm tra nhanh

1. Boosting giảm bias hay variance? Vì sao dùng weak learner (stump) thay vì cây sâu?
2. Một stump có lỗi có trọng số $\varepsilon = 0.3$. $\alpha$ bằng bao nhiêu? Trọng số các điểm bị sai nhân với bao nhiêu?
3. Điều gì xảy ra nếu một weak learner có $\varepsilon > 0.5$?
4. Vì sao AdaBoost nhạy với nhiễu nhãn? Cách giảm thiểu?

## 📌 Tóm tắt

- Boosting: tuần tự, model sau sửa lỗi model trước; giảm bias; có thể overfit.
- AdaBoost: tăng trọng số mẫu sai, model tốt có $\alpha$ lớn; lỗi train giảm theo hàm mũ.
- AdaBoost = forward stagewise tối thiểu exponential loss → nhạy với nhiễu nhãn (học thuộc điểm sai khi nhiều vòng); tổng quát hoá → Gradient Boosting.

## 📚 Tài liệu tham khảo

- Freund & Schapire (1997) — *A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting*
- Friedman, Hastie & Tibshirani (2000) — *Additive Logistic Regression: a Statistical View of Boosting*
- ESL §10.1–10.6